<a href="https://colab.research.google.com/github/jiraroj-wir/MUIC-ICCS261-Principles-of-Data-Science/blob/main/Copy_of_M2P2_Data_and_Sampling_Student_Notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M2P2 Data and Sampling — Student Notes

*Generated by Claude based on the class slides*

**ICCS261, Module 2, Part 2.** So far we have described the data we have (EDA). This module is about reasoning from a sample to the population we don't have. Run each code cell as you read (**Shift + Enter**) and compare your output with the numbers in the comments.

**After this lecture you should be able to:**

1. Tell a population from a sample, and pick simple random, with/without replacement, or stratified sampling.
2. Spot sample bias, self-selection bias, selection bias (data snooping, vast search effect) and regression to the mean.
3. Explain a sampling distribution, the Central Limit Theorem and the standard error.
4. Use the bootstrap to estimate the variability of any statistic.
5. Build and interpret a confidence interval, and say what drives its width.
6. Choose a suitable probability distribution: normal, t, binomial, chi-square, F, Poisson, exponential or Weibull.

> **How to study with these notes.** Run the cells **in order** — later cells reuse variables from earlier ones. Some cells use random sampling, so your numbers may differ slightly; the lesson should not. Try each *Check your understanding* box before moving on, and write your answers in the cell below it. We discuss the answers in class.

## Setup — run this first

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

titanic = sns.load_dataset("titanic")   # 891 passengers, 15 columns
rng = np.random.default_rng(261)         # fixed seed so results are reproducible
titanic.head()

## 1. Sampling principles
*Slides 5–15*

We rarely see the whole population, so we study a sample and reason back to it. Think of an election poll: a few thousand respondents can speak for millions of voters — if they are chosen well.

**Population vs sample**

- A **sample** is a subset of data drawn from a larger dataset, the **population**.
- The population is often *theoretical* — e.g. "all passengers who could ever board a ship like the Titanic".
- Notation: $\bar{x}$ ("x-bar") is the sample mean; $\mu$ is the population mean. We use $\bar{x}$ to estimate $\mu$.

**Random sampling** — every member has an equal chance of selection.

| Method | What happens | Typical use |
| --- | --- | --- |
| Without replacement | A selected item leaves the pool; it cannot be picked twice | Most surveys, e.g. calling 500 of 10,000 voters once each |
| With replacement | A selected item goes back; it can be picked again | Bootstrap, multiple imputation |
| Stratified | Split the population into homogeneous **strata**, then sample randomly within each | Guarantees small groups are represented |

**Bias** is *systematic* error, unlike random error from chance.

- **Sample bias:** the sample differs from the population in a meaningful, nonrandom way.
- **Self-selection bias:** people choose to be in the sample (online reviews, opt-in polls).
- **Literary Digest, 1936:** 10 million responses drawn from car and phone owners got the US election wrong. Gallup, with about 2,000 scientifically chosen people, got it right. **Quality (representativeness) beats quantity.**

### Try it — three ways to sample 100 Titanic passengers
Treat the 891 passengers as the population, so we know the true answer.

In [ ]:
pop_rate = titanic["survived"].mean()                                  # 0.384
srs  = titanic.sample(n=100, replace=False, random_state=1)            # simple random, no replacement
boot = titanic.sample(n=100, replace=True,  random_state=1)            # with replacement
print(round(pop_rate, 3), srs["survived"].mean(), boot["survived"].mean())   # 0.384, 0.39, 0.33
print("repeated rows:", boot.index.duplicated().sum())                 # 3 passengers drawn twice

# Stratified by class: about 11% of every class
strat = titanic.groupby("pclass", group_keys=False).sample(frac=100/891, random_state=1)

pd.DataFrame({
    "population":    titanic["pclass"].value_counts(normalize=True),
    "simple random": srs["pclass"].value_counts(normalize=True),
    "stratified":    strat["pclass"].value_counts(normalize=True),
}).round(3)

Stratified sampling matches the class mix almost exactly (0.55 / 0.24 / 0.21). The simple random sample is close but drifts. Change `random_state=1` to `2, 3, …` and re-run to see how much a simple random sample can wander.

> **Check your understanding 1**
>
> 1. Is the Titanic table a population or a sample? Defend both answers.
> 2. You survey students by posting a link in the class LINE group. Name the bias.
> 3. Brain teaser (slide 13): you are studying health habits across Thailand. List three factors you would use as strata, and why.
> 4. Why did 2,000 Gallup responses beat 10 million Literary Digest responses?

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## 2. Selection bias
*Slides 16–23*

Selection bias is choosing data — consciously or not — in a way that leads to a misleading or short-lived conclusion.

- **Data snooping:** hunting through data for something interesting. "If you torture the data long enough, sooner or later it will confess."
- **Vast search effect:** run enough models or ask enough questions and one will look significant by chance. One person flipping 10 heads in a row is impressive only if predicted beforehand; in a stadium of 20,000 coin-flippers, someone will do it by luck. **Fix:** validate on a holdout set the search never touched.
- **Regression to the mean:** extreme observations tend to be followed by more central ones. The bottom 10% of test-takers improve on a retest even without the new math software, partly because their bad luck fades. Don't credit an intervention for what is just data stabilising.
- **Typical forms:** nonrandom sampling, cherry-picking, stopping an experiment early when results look "interesting". **Protection:** state the hypothesis first, then collect data with randomisation.

### Try it A — a convenient subset is a biased subset
Only 203 of 891 passengers have a recorded `deck`. What if we analysed only them?

In [ ]:
known_deck = titanic[titanic["deck"].notna()]
print(len(known_deck))                                                   # 203
print(known_deck["survived"].mean(), titanic["survived"].mean())         # 0.670 vs 0.384
known_deck["pclass"].value_counts(normalize=True)                        # 86% first class!

The "known deck" group is 86% first class, so its survival rate is wildly optimistic. Missingness is not random here.

### Try it B — the vast search effect
Add 200 columns of pure random noise, then search for the one that "predicts" survival best.

In [ ]:
noise = pd.DataFrame(rng.normal(size=(len(titanic), 200)),
                     columns=[f"feature_{i}" for i in range(200)])
corrs = noise.corrwith(titanic["survived"]).abs().sort_values(ascending=False)
best = corrs.index[0]
r, p = stats.pearsonr(noise[best], titanic["survived"])
print(best, round(r, 3), round(p, 4))          # e.g. r ≈ 0.10, p ≈ 0.005 -- "significant"!

n_sig = sum(stats.pearsonr(noise[c], titanic["survived"])[1] < 0.05 for c in noise)
print(n_sig, "of 200 noise features have p < 0.05")

# Holdout check: does the 'winner' survive on rows it was not chosen from?
r_train = np.corrcoef(noise[best][:600], titanic["survived"][:600])[0, 1]
r_hold  = np.corrcoef(noise[best][600:], titanic["survived"][600:])[0, 1]
print(round(r_train, 3), round(r_hold, 3))

Count how many noise columns come out "significant", and watch what happens to the winner's correlation on the holdout rows.

**Titanic brain teaser (slide 23).** Why don't we explore how `deck` relates to everything else? Are we cherry-picking? Look back at Try it A before you answer.

> **Check your understanding 2**
>
> 1. A colleague tests 50 features and reports the one with p = 0.03. What should you ask?
> 2. A store's worst-performing month is followed by a new manager and a sales rise. Name a non-causal explanation.
> 3. Why does a holdout set protect against the vast search effect?
> 4. In Try it B, roughly how many of 200 pure-noise features should have p < 0.05? Why?

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## 3. Sampling distributions, CLT, standard error and the bootstrap
*Slides 24–38*

**Descriptive vs inferential statistics.** Descriptive statistics summarise the data at hand ("the average delay of these 100 flights is 12 min"). Inferential statistics generalise to the population, with probability-based conclusions ("the average delay on this route is likely 10–14 min"). Probability runs from a known population to data; statistics runs from data back to the population.

**Data distribution vs sampling distribution**

- **Data distribution:** the values of individual observations (each passenger's fare).
- **Sampling distribution:** the values of a *statistic* (e.g. the mean) across many samples from the same population.
- The sampling distribution is narrower and more bell-shaped than the raw data.
- Recipe: fix n → draw many random samples → compute the statistic for each → plot those statistics.

**Central Limit Theorem (CLT).** As n grows, the sampling distribution of the mean becomes approximately normal and centred on $\mu$ — even when the raw data is not normal. n ≥ 30 is a common rule of thumb, unless the data is very far from normal. The CLT is why z- and t-based formulas work.

**Standard error (SE)** measures how precisely a statistic estimates its parameter. Standard deviation describes individual values; standard error describes the statistic.

$$SE = \frac{s}{\sqrt{n}}$$

Because of the $\sqrt{n}$, halving the SE needs 4× the sample size. Notation: $s$ is the sample SD; $\sigma$ is the population SD.

### Try it — the CLT on a very skewed variable
`fare` is heavily right-skewed: mean 32.20, median 14.45, skewness 4.8. Like the LendingClub example (slide 32), we draw 1,000 samples at several sizes.

In [ ]:
fare = titanic["fare"]
rows = []
for n in [1, 5, 20, 50]:
    means = [fare.sample(n, replace=True).mean() for _ in range(1000)]
    rows.append(pd.DataFrame({"sample mean": means, "n": f"n = {n}"}))
    print(f"n={n:>2}: SD of means = {np.std(means, ddof=1):6.2f}   s/sqrt(n) = {fare.std()/np.sqrt(n):6.2f}")

sims = pd.concat(rows)
sns.displot(data=sims, x="sample mean", col="n", col_wrap=2, bins=40,
            facet_kws={"sharex": False, "sharey": False}, height=2.6);

| n | SD of 1,000 sample means | s / √n (theory) |
| --- | --- | --- |
| 1 | 58.8 | 49.7 |
| 5 | 22.2 | 22.2 |
| 20 | 10.6 | 11.1 |
| 50 | 7.0 | 7.0 |

At n = 1 the histogram is just the skewed fare distribution. By n = 50 it is close to a bell curve, and its spread matches $s/\sqrt{n}$.

**The bootstrap** (slides 36–38). What if n is small, the statistic is not a mean (a median, a ratio, a model coefficient), or you cannot draw new samples? Resample **your own sample with replacement**, many times, and recompute the statistic each time. The spread of those values estimates the SE; their percentiles give a confidence interval. It needs no normality assumption and works for almost any statistic.

In [ ]:
def bootstrap(data, stat, B=5000):
    data = np.asarray(data)
    return np.array([stat(rng.choice(data, size=len(data), replace=True)) for _ in range(B)])

boot_medians = bootstrap(fare, np.median)
print("bootstrap SE:", boot_medians.std(ddof=1))           # ≈ 0.74
print("95% CI:", np.percentile(boot_medians, [2.5, 97.5]))  # ≈ [13.00, 15.55]
sns.histplot(boot_medians, bins=40);

In [ ]:
# Same result with SciPy's built-in
stats.bootstrap((fare.to_numpy(),), np.median, n_resamples=5000, method="percentile").confidence_interval

The CLT gives a clean formula for the mean; the bootstrap gives us the median fare's uncertainty with no formula at all.

> **Check your understanding 3**
>
> 1. In one sentence, distinguish the distribution of `fare` from the sampling distribution of mean `fare`.
> 2. Fare is very skewed. Why does the histogram of sample means still look normal at n = 50?
> 3. Your SE is 4. How large must n become to get SE = 2?
> 4. Why must bootstrap resampling be *with* replacement? What happens without it?
> 5. Name a statistic where the bootstrap is more convenient than a CLT formula.

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## 4. Confidence intervals
*Slides 39–52*

A single point estimate ("mean age is 29.7") is misleadingly precise. A confidence interval gives a range that reflects sampling variability.

- **Meaning:** a 95% CI comes from a *procedure* that captures the true parameter in about 95% of repeated samples. The 95% describes the method, not this one interval.
- **True value:** the error-free population parameter. True score theory: observed x = true value t + error e. We rarely know t, so we bracket it.
- **Two-sided vs one-sided:** two-sided splits $\alpha$ across both tails (standard). One-sided puts all of $\alpha$ in one tail when only one direction matters.

**General recipe**

$$CI = \text{point estimate} \pm (\text{critical value} \times SE) \qquad\text{e.g.}\qquad \bar{x} \pm z^{*}\,\frac{s}{\sqrt{n}}$$

1. Choose the distribution: z if n ≥ 30, otherwise t with df = n − 1.
2. Choose the confidence level CL (90%, 95%, 99%); α = 1 − CL.
3. Two-sided: look up the value leaving α/2 in each tail (95% → 0.025 per tail → z* = 1.96).
4. Compute estimate ± critical value × SE.

**What controls the width**

- **More data → narrower** (SE shrinks with √n).
- **Higher confidence → wider.** To be more confident you accept more uncertainty; to be more precise you lower confidence or collect more data.
- **Uses:** show precision (two means of 10: [5, 15] is far less precise than [9, 11]); compare groups (non-overlapping 95% CIs ⇒ a significant difference); decide whether you need more data.
- **Non-normal data:** fit a matching distribution (e.g. log-normal, then back-transform the bounds), or use the bootstrap (Section 3).

### Try it A — 95% CI for mean age

In [ ]:
age = titanic["age"].dropna()
n, xbar, s = len(age), age.mean(), age.std()      # 714, 29.70, 14.53
se = s / np.sqrt(n)                               # 0.544
z = stats.norm.ppf(0.975)                         # 1.960
t = stats.t.ppf(0.975, df=n - 1)                  # 1.963
print(xbar - z*se, xbar + z*se)                   # [28.63, 30.76]

for cl in [0.90, 0.95, 0.99]:
    zc = stats.norm.ppf(1 - (1 - cl) / 2)
    print(f"{cl:.0%} CI width: {2*zc*se:.2f} years")   # 1.79, 2.13, 2.80

small = age.sample(25, random_state=3)            # what if we only had 25 passengers?
stats.t.interval(0.95, df=24, loc=small.mean(), scale=small.std()/np.sqrt(25))  # [22.3, 35.7]

Same population, n = 25 instead of 714: the interval grows from about 2 years wide to about 13.

### Try it B — comparing groups with CIs

In [ ]:
def prop_ci(x, cl=0.95):
    p, n = x.mean(), len(x)
    m = stats.norm.ppf(1 - (1 - cl) / 2) * np.sqrt(p * (1 - p) / n)
    return pd.Series({"n": n, "rate": p, "low": p - m, "high": p + m})

display(titanic.groupby("pclass")["survived"].apply(prop_ci).unstack().round(3))
sns.pointplot(data=titanic, x="pclass", y="survived", hue="sex", errorbar=("ci", 95), dodge=True);

| Class | n | Survival rate | 95% CI |
| --- | --- | --- | --- |
| 1st | 216 | 0.630 | 0.565 – 0.694 |
| 2nd | 184 | 0.473 | 0.401 – 0.545 |
| 3rd | 491 | 0.242 | 0.204 – 0.280 |

None of the three intervals overlap, so the class differences are statistically significant. By sex the gap is larger still: female 0.742 (0.694–0.790) vs male 0.189 (0.157–0.221).

> **Check your understanding 4**
>
> 1. True or false: "There is a 95% probability that the true mean age is between 28.63 and 30.76." Fix the wording.
> 2. Moving from 95% to 99% confidence made the age CI wider. Why must it?
> 3. Why is the 2nd-class interval wider than the 3rd-class interval?
> 4. Two groups' 95% CIs overlap slightly. Can you conclude there is no difference?
> 5. When would you choose t over z? Why do they barely differ for mean age?

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## 5. Probability distributions
*Slides 53–87*

A probability distribution says how likely each outcome of a random variable is. We use distributions to describe variability, predict behaviour (e.g. staffing for customer arrivals), support decisions (financial risk) and simulate scenarios (clinical trials). Most tests and CIs assume one.

| Distribution | Type | Models | Titanic / slide example |
| --- | --- | --- | --- |
| Normal | Continuous | Symmetric measurements; sampling distributions and errors | `age` (roughly) |
| Long-tailed | Either | Extreme values more often than the normal predicts | `fare` |
| Student's t | Continuous | Mean with small n or unknown σ | Mean age of 10 first-class passengers |
| Binomial | Discrete | Successes in n yes/no trials | Survivors in a group of 10 |
| Chi-square | Counts in categories | Observed vs expected counts; independence | sex × survived |
| F | Continuous | Between-group vs within-group variance (ANOVA, regression) | fare across classes |
| Poisson | Discrete | Number of events per interval, rate λ | 3 flight delays per hour |
| Exponential | Continuous | Time between events, same λ | Waiting time to the next delay |
| Weibull | Continuous | Time to failure when the rate changes over time | Machine wear-out |

**Normal and the z-score.** Defined by $\mu$ (centre) and $\sigma$ (spread); probabilities are areas under the curve. Standardise with $z = (x - \mu)/\sigma$, then $P(X < x) = \Phi(z)$. About 68% / 95% / 99.7% of values fall within 1 / 2 / 3 SDs. Slide 63: scores with μ = 100, σ = 15 → P(X < 120) = Φ(1.33) ≈ 0.908. A **QQ-plot** checks normality: points on the line mean normal; curving tails mean skew or long tails.

**Long tails.** Real data is often skewed or long-tailed. Rare extreme events ("black swans" — 2008 crisis, COVID-19) happen far more often than a normal model predicts.

In [ ]:
z_age = (age - age.mean()) / age.std()
print([round((z_age.abs() <= k).mean(), 3) for k in (1, 2, 3)])   # [0.72, 0.96, 0.997] -- close to 68-95-99.7
print(stats.norm.cdf(20, loc=age.mean(), scale=age.std()))        # model: 0.25 under 20; actual: 0.23
print((fare > fare.mean() + 3 * fare.std()).sum())                 # 20 passengers; a normal model expects ~1.2

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
stats.probplot(age,  dist="norm", plot=axes[0]); axes[0].set_title("age")
stats.probplot(fare, dist="norm", plot=axes[1]); axes[1].set_title("fare");

**Student's t** (Gosset, Guinness brewery). Heavier tails than the normal, set by df = n − 1; use it for small samples or unknown σ. It approaches the normal as n grows.

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$$

In [ ]:
first = titanic.loc[titanic["pclass"] == 1, "age"].dropna().sample(10, random_state=7)
t = (first.mean() - 30) / (first.std() / np.sqrt(10))   # mean 34.8, s 10.0 -> t = 1.52
print(t, 1 - stats.t.cdf(t, df=9))                       # P(T > t) = 0.081
stats.ttest_1samp(first, 30, alternative="greater")      # same answer in one line

**Binomial** — x successes in n independent trials, each with probability p. Slide 75: 3 heads in 5 fair flips = 0.3125.

$$P(X = x) = \binom{n}{x}\, p^{x} (1 - p)^{n - x}$$

In [ ]:
p = titanic["survived"].mean()                  # 0.384
print(stats.binom.pmf(4, n=10, p=p))            # P(exactly 4 of 10 random passengers survive) = 0.249
print(stats.binom.cdf(2, n=10, p=p))            # P(at most 2 survive) = 0.195

**Chi-square** — how far observed counts depart from expected counts. df = (rows − 1)(cols − 1), or categories − 1 for goodness of fit.

$$\chi^{2} = \sum_i \frac{(O_i - E_i)^{2}}{E_i}$$

In [ ]:
print(stats.chisquare([50, 60, 40, 47, 53]))   # slide 77 store: chi2 = 4.36, p = 0.36

table = pd.crosstab(titanic["sex"], titanic["survived"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(chi2, dof, p)                            # 260.7, 1, 1.2e-58
print(expected.round(1))                       # if independent, ~120 women would survive; 233 did

**F** — ratio of between-group variability to within-group variability. Used in ANOVA (do several group means differ?) and regression. A larger F means the factor explains more than random noise.

In [ ]:
groups = [g["fare"].values for _, g in titanic.groupby("pclass")]
stats.f_oneway(*groups)                        # F = 242.3, p ≈ 1e-84: fares differ by class

**Poisson, exponential, Weibull.** Poisson counts events per interval at a constant rate λ; exponential is the time between those events with the same λ. Weibull generalises the exponential when the rate changes: shape β > 1 means failures become more likely over time. The Titanic has no event-time column, so we use the slide 57 airline example.

In [ ]:
print(stats.poisson.pmf(5, mu=3))              # P(exactly 5 delays in an hour | 3 per hour) = 0.101
print(stats.expon.sf(0.5, scale=1/3))          # P(no delay for 30+ minutes) = 0.223

**Choosing a distribution** (slides 86–87): check the data type (counts → discrete, measurements → continuous); plot it (histogram, QQ-plot); test goodness of fit (Kolmogorov–Smirnov, chi-square); then apply theory and domain knowledge.

> **Check your understanding 5**
>
> 1. Match each to a distribution: (a) number of emails per hour, (b) minutes until the next email, (c) whether each of 20 users clicks an ad, (d) heights of 1,000 adults.
> 2. Based on its QQ-plot and the "20 vs 1.2" count, should you model `fare` as normal? What could you do instead?
> 3. Why do we use t rather than z for the 10 first-class passengers?
> 4. In the sex × survived table, what does a huge χ² tell us, and what does it *not* tell us?
> 5. A machine's failure risk rises with age. Exponential or Weibull? Which parameter shows this?

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## Case study: sampling to impute missing Titanic ages
*Slide 22 — hands-on*

177 of 891 ages (20%) are missing. Filling them all with the mean is quick, but it shrinks the spread and piles everyone onto one value. Instead, we fill each gap with a **random draw, with replacement**, from observed ages of passengers in the same class and sex — sampling within strata.

In [ ]:
titanic.isna().sum()[lambda s: s > 0]      # age 177, deck 688, embarked 2, embark_town 2

In [ ]:
def impute_by_sampling(df, col, by, rng):
    """Fill missing values in `col` with random draws from the same stratum."""
    out = df[col].copy()
    for _, idx in df.groupby(by, observed=True).groups.items():
        grp = df.loc[idx, col]
        missing = grp.index[grp.isna()]
        if len(missing):
            out.loc[missing] = rng.choice(grp.dropna().to_numpy(), size=len(missing), replace=True)
    return out

titanic["age_mean"] = titanic["age"].fillna(titanic["age"].mean())
titanic["age_sampled"] = impute_by_sampling(titanic, "age", ["pclass", "sex"], rng)

display(titanic[["age", "age_mean", "age_sampled"]].agg(["mean", "std", "median"]).round(2))

long = titanic[["age", "age_mean", "age_sampled"]].melt(var_name="version", value_name="value")
sns.kdeplot(data=long, x="value", hue="version", common_norm=False);

| Version | Mean | SD | Median |
| --- | --- | --- | --- |
| Observed ages only (n = 714) | 29.70 | 14.53 | 28.0 |
| Mean imputation | 29.70 | 13.00 | 29.7 |
| Stratified sampling imputation | 29.40 | 14.45 | 28.0 |

Mean imputation keeps the mean but cuts the SD by about 10% and creates a spike at 29.7 in the KDE plot. Sampling imputation keeps the shape. Because it is random, each run gives slightly different values — repeating it several times and combining results is the idea behind **multiple imputation**.

### Extension tasks
Write your code in the empty cells below.

1. Impute without strata: fill every gap with `rng.choice` from all observed ages. Is the mean closer or further from 29.70? Why might class and sex matter?
2. Re-run the imputation 20 times and plot the 20 means of `age_sampled`. How much does imputation alone move the mean?
3. Compute a 95% CI for mean age using `age_mean`. Why is it misleadingly narrow?
4. Should we impute `deck` the same way? Compare `pclass` for rows with and without a deck before you answer.

In [ ]:
# Extension task 1

In [ ]:
# Extension task 2

In [ ]:
# Extension task 3

In [ ]:
# Extension task 4

> **Check your understanding — case study**
>
> 1. Why does mean imputation shrink the standard deviation?
> 2. Why sample *with* replacement from the observed ages?
> 3. What assumption about *why* ages are missing does this method make?

**✏️ Your answers** — double-click this cell to write.

1.
2.
3.
4.

## Summary

Collect data randomly (and stratify when groups matter), beware searching until something looks interesting, and report estimates with their uncertainty — via the CLT formula when it applies, and the bootstrap when it doesn't.

**Key terms**

| Term | Meaning |
| --- | --- |
| Population / sample | The full (often theoretical) set / the subset we observe |
| Bias | Systematic error, as opposed to random error |
| Stratified sampling | Random sampling within homogeneous subgroups |
| Vast search effect | Finding "significant" patterns by chance after many tries |
| Regression to the mean | Extreme values tend to be followed by more central ones |
| Sampling distribution | Distribution of a statistic across repeated samples |
| Central Limit Theorem | Sample means become approximately normal as n grows |
| Standard error | SD of a statistic; s/√n for the mean |
| Bootstrap | Resampling your sample with replacement to see how a statistic varies |
| Confidence interval | Point estimate ± critical value × SE |
| z-score | (x − μ) / σ: distance from the mean in SDs |
| Degrees of freedom | n − 1 for a one-sample t |

**Self-check**

1. A website shows the average rating from users who chose to leave a review. Name the bias, and suggest a better sampling plan.
2. Doubling the sample size reduces the standard error by what factor?
3. Write the 95% CI formula for a mean, and name what each symbol is.
4. Which distribution would you use for the number of Titanic survivors in a random group of 20 passengers? Give its parameters.

**Looking ahead.** Why is α = 0.05 the usual cut-off for "significant"? It is a convention popularised by Ronald Fisher in the 1920s, not a law. Next session we start hypothesis testing and see how to choose α by the cost of being wrong.